In [1]:
import sys, os
# My dot keeps going missing
os.environ["PATH"] = "/opt/homebrew/Cellar/graphviz/12.2.1/bin/:" + os.environ["PATH"]
sys.path.insert(0, os.path.abspath(".."))
import polars as pl
from ped import initialize_ped
initialize_ped() # Imports all the submodules

from mixed_types import MixedTypesModule
from ped.modules.credit.scorecard.module import ScoreCard, ScoredVariable
from ped.modules.credit.scorecard.impl import BoundBin, ValuesBin, DefaultBin
from ped.flow import FlowConfiguration

/opt/miniconda/envs/dspdev/lib/python3.10/site-packages/pyspark/pandas/__init__.py:43: UserWarning: 'PYARROW_IGNORE_TIMEZONE' environment variable was not set. It is required to set this environment variable to '1' in both driver and executor sides if you use pyarrow>=2.0.0. pandas-on-Spark will set it for you but it does not work if there is a Spark context already launched.
  warnings.warn(
/opt/miniconda/envs/dspdev/lib/python3.10/site-packages/pydantic/_internal/_fields.py:198: UserWarning: Field name "schema" in "DataFrame" shadows an attribute in parent "BaseModel"
  warnings.warn(
/opt/miniconda/envs/dspdev/lib/python3.10/site-packages/pydantic/_internal/_fields.py:198: UserWarning: Field name "schema" in "WithTreeOutput" shadows an attribute in parent "BaseModel"
  warnings.warn(


In [2]:
data = pl.LazyFrame({
    "customer_id":          [1,        2,        3,        4,        5       ],
    "income":               [30000,    75000,    50000,    120000,   45000   ],
    "debt_to_income":       [0.6,      0.2,      0.35,     0.1,      0.5     ],
    "credit_history_years": [1,        12,       5,        20,       3       ],
    "employment_status":    ["part_time", "full_time", "full_time", "full_time", "unemployed"],
})

data.collect()

customer_id,income,debt_to_income,credit_history_years,employment_status
i64,i64,f64,i64,str
1,30000,0.6,1,"""part_time"""
2,75000,0.2,12,"""full_time"""
3,50000,0.35,5,"""full_time"""
4,120000,0.1,20,"""full_time"""
5,45000,0.5,3,"""unemployed"""


In [3]:


scorecard = ScoreCard(
    name="my_scorecard",
    variables=[
        ScoredVariable(
            name="income_var",
            variable_name="income",
            bins=[
                BoundBin(value=50,  lower_bound=None,   upper_bound=30000),
                BoundBin(value=100, lower_bound=30000,  upper_bound=60000),
                BoundBin(value=150, lower_bound=60000,  upper_bound=100000),
                BoundBin(value=200, lower_bound=100000, upper_bound=None),
            ],
            default=DefaultBin(value=75),
            value_output_name="income_score",
        ),
        ScoredVariable(
            name="dti_var",
            variable_name="debt_to_income",
            bins=[
                BoundBin(value=200, lower_bound=None, upper_bound=0.2),
                BoundBin(value=150, lower_bound=0.2,  upper_bound=0.4),
                BoundBin(value=100, lower_bound=0.4,  upper_bound=0.6),
                BoundBin(value=50,  lower_bound=0.6,  upper_bound=None),
            ],
            default=DefaultBin(value=100),
            value_output_name="dti_score",
        ),
        ScoredVariable(
            name="employment_var",
            variable_name="employment_status",
            bins=[
                ValuesBin(value=150, items=["full_time"]),
                ValuesBin(value=100, items=["part_time"]),
                ValuesBin(value=50,  items=["unemployed"]),
            ],
            default=DefaultBin(value=75),
            value_output_name="employment_score",
        ),
    ],
    score_output_name="score",
)

In [4]:
mixed_types_module = MixedTypesModule(
    name="mtm", 
    input_mapping={"spend": "my_scorecard.score", "signups": "credit_history_years"}
)

In [6]:

flow_from_ext = FlowConfiguration(
    metadata={
        "name": "external_example_flow",
        "author": "i must make this optional",
        "description": "i must make this optional",
    },
    modules=[
        scorecard.model_dump(),
        mixed_types_module.model_dump(),
    ],
    outputs=["custom_example.eg3.c"],
)
loaded_modules = await flow_from_ext.get_parameterized_modules(inputs={})
graph_ext = await flow_from_ext.build_graph(inputs={})
graph_ext.hamilton_driver

-------------------------------------------------------------------
Oh no an error! Need help with Hamilton?
Join our slack and ask for help! https://join.slack.com/t/hamilton-opensource/shared_invite/zt-2niepkra8-DGKGf_tTYhXuJWBTXtIs4g
-------------------------------------------------------------------



ValueError: Error: mtm.avg_3wk_spend is expecting my_scorecard.score:<class 'pandas.core.series.Series'>, but found my_scorecard.score:<class 'polars.expr.expr.Expr'>. 
Hamilton does not consider these types to be equivalent. If you believe they are equivalent, please reach out to the developers. Note that, if you have types that are equivalent for your purposes, you can create a graph adapter that checks the types against each other in a more lenient manner.

In [ ]:
result_ext = graph_ext.execute(
    inputs={
        "input": data,
    },
)
# I expect result_ext to be some lazy frame

In [ ]:
result_ext = graph_ext.execute(
    inputs={
        "input": data.collect().to_pandas(),
    },
)
# I expect result_ext to be a dataframe